In [ ]:
# 导入库
import pandas as pd
import numpy as np
import re

# 汇率设置：1美元=7.2人民币
EXCHANGE_RATE = 7.2

In [ ]:
# movie1标准字段
standard_columns = [
    'color', 'director_name', 'num_critic_for_reviews', 'duration',
    'director_facebook_likes', 'actor_3_facebook_likes', 'actor_2_name',
    'actor_1_facebook_likes', 'gross', 'genres', 'actor_1_name',
    'movie_title', 'num_voted_users', 'cast_total_facebook_likes',
    'actor_3_name', 'facenumber_in_poster', 'plot_keywords',
    'movie_imdb_link', 'num_user_for_reviews', 'language', 'country',
    'content_rating', 'budget', 'title_year', 'actor_2_facebook_likes',
    'imdb_score', 'aspect_ratio', 'movie_facebook_likes'
]


In [ ]:
# 票房转换函数 - 核心修正
def convert_gross_to_usd(value):
    """将票房数据转换为美元（修复单位换算错误）"""
    if pd.isna(value) or value == '':
        return np.nan
    value_str = str(value).strip()
    if value_str == '--' or value_str == '':
        return np.nan
    
    # 提取数字和单位
    match = re.match(r'([\d\.]+)\s*([亿万]?)', value_str)
    if not match:
        try:
            return float(value_str) / EXCHANGE_RATE
        except:
            return np.nan
    
    num = float(match.group(1))
    unit = match.group(2)
    
    # 关键修正：正确单位换算
    if unit == '亿':
        value_yuan = num * 100000000  # 亿转元
    elif unit == '万':
        value_yuan = num * 10000      # 万转元
    else:
        value_yuan = num              # 无单位默认为元
    
    # 人民币元转换为美元
    return value_yuan / EXCHANGE_RATE

# 国家名称转换函数 - 将中文国家名称转换为英文格式
def convert_country_to_english(country_str):
    """
    将中文国家名称转换为英文格式
    
    功能说明：
    1. 空值保持不变（None 或空字符串）
    2. 已经是英文的保持不变
    3. 包含逗号的分割处理（多个国家用逗号分隔）
    4. 使用映射表转换常见国家
    
    参数：
    country_str : str 或 None
        待转换的国家名称字符串，可以是中文、英文或包含逗号分隔的多个国家
        
    返回值：
    str 或 None
        转换后的英文国家名称字符串，如果输入为空则返回原值
    """
    
    # 处理空值情况
    if country_str is None:
        return None
        
    # 转换为字符串类型，确保处理非字符串输入
    country_str = str(country_str)
    
    # 去除首尾空格
    country_str = country_str.strip()
    
    # 如果是空字符串，直接返回
    if country_str == '':
        return ''
    
    # 定义国家名称映射表（中文 -> 英文）
    country_mapping = {
        # 用户指定的映射
        "中国大陆": "China",
        "中国": "China",
        "美国": "USA",
        "日本": "Japan",
        "英国": "UK",
        "爱尔兰": "Ireland",
        "中国香港": "Hong Kong",
        "香港": "Hong Kong",
        "意大利": "Italy",
        "加拿大": "Canada",
        "法国": "France",
        "德国": "Germany",
        # 补充其他常见国家映射
        "澳大利亚": "Australia",
        "新西兰": "New Zealand",
        "韩国": "South Korea",
        "新加坡": "Singapore",
        "印度": "India",
        "俄罗斯": "Russia",
        "巴西": "Brazil",
        "墨西哥": "Mexico",
        "西班牙": "Spain",
        "葡萄牙": "Portugal",
        "荷兰": "Netherlands",
        "瑞士": "Switzerland",
        "瑞典": "Sweden",
        "挪威": "Norway",
        "丹麦": "Denmark",
        "芬兰": "Finland",
        "奥地利": "Austria",
        "比利时": "Belgium",
        "波兰": "Poland",
        "泰国": "Thailand",
        "越南": "Vietnam",
        "马来西亚": "Malaysia",
        "印度尼西亚": "Indonesia",
        "菲律宾": "Philippines",
        "沙特阿拉伯": "Saudi Arabia",
        "阿联酋": "United Arab Emirates",
        "土耳其": "Turkey",
        "埃及": "Egypt",
        "南非": "South Africa",
    }
    
    # 将中文逗号替换为英文逗号，统一分隔符
    country_str = country_str.replace('，', ',')
    
    # 检查是否包含逗号（多个国家）
    if ',' in country_str:
        # 分割字符串，处理每个部分
        parts = country_str.split(',')
        converted_parts = []
        
        for part in parts:
            part = part.strip()
            if part == '':
                # 空部分保留
                converted_parts.append('')
                continue
                
            # 检查是否为中文（在映射表中）
            if part in country_mapping:
                converted_parts.append(country_mapping[part])
            else:
                # 不在映射表中，假设已经是英文，原样保留
                converted_parts.append(part)
        
        # 用逗号重新连接
        return ','.join(converted_parts)
    
    # 单个国家处理
    # 检查是否在映射表中
    if country_str in country_mapping:
        return country_mapping[country_str]
    
    # 不在映射表中，假设已经是英文，原样返回
    return country_str

# 电影类型转换函数 - 将中文电影类型转换为英文格式（|分隔）
def convert_genres_to_english(genres_str):
    """
    将中文电影类型转换为英文格式，统一为|分隔
    
    功能说明：
    1. 空值保持不变（None 或空字符串）
    2. 已经是英文的保持不变（|分隔）
    3. 支持多种分隔符：／、逗号、中文逗号
    4. 使用映射表转换常见电影类型
    
    参数：
    genres_str : str 或 None
        待转换的电影类型字符串，可以是中文、英文或包含不同分隔符的多个类型
        
    返回值：
    str 或 None
        转换后的英文电影类型字符串（|分隔），如果输入为空则返回原值
    """
    
    # 处理空值情况
    if genres_str is None:
        return None
        
    # 转换为字符串类型，确保处理非字符串输入
    genres_str = str(genres_str)
    
    # 去除首尾空格
    genres_str = genres_str.strip()
    
    # 如果是空字符串，直接返回
    if genres_str == '':
        return ''
    
    # 定义电影类型映射表（中文 -> 英文）
    genres_mapping = {
        # movie2和movie3中出现的类型
        "动画": "Animation",
        "喜剧": "Comedy",
        "奇幻": "Fantasy",
        "剧情": "Drama",
        "运动": "Sport",
        "体育": "Sport",  # 同义词
        "家庭": "Family",
        "冒险": "Adventure",
        "历史": "History",
        "战争": "War",
        "爱情": "Romance",
        "青春": "Youth",
        "灾难": "Disaster",
        "动作": "Action",
        "犯罪": "Crime",
        "音乐": "Music",
        "传记": "Biography",
        "纪录片": "Documentary",
        "纪录": "Documentary",  # 简写
        "悬疑": "Mystery",
        "古装": "Costume",
        "科幻": "Sci-Fi",
        "惊悚": "Thriller",
        "合家欢": "Family",
        "3D": "3D",
        "IMAX": "IMAX",
        "2D": "2D",
        "3DIMAX": "3D IMAX",
        "2DIMAX": "2D IMAX",
        # 补充其他常见类型
        "恐怖": "Horror",
        "儿童": "Children",
        "歌舞": "Musical",
        "武侠": "Martial Arts",
        "警匪": "Police",
        "黑色幽默": "Black Comedy",
        "超级英雄": "Superhero",
        "魔幻": "Magic",
        "神话": "Mythology",
        "史诗": "Epic",
        "西部": "Western",
        "cult": "Cult",
        "独立": "Indie",
        "短片": "Short",
        "微电影": "Short Film",
    }
    
    # 去除多余的空白字符和换行
    genres_str = ' '.join(genres_str.split())
    
    # 如果已经是英文格式（包含|分隔符），直接返回
    # 检查是否包含中文字符
    def contains_chinese(text):
        return any('\u4e00' <= char <= '\u9fff' for char in text)
    
    if not contains_chinese(genres_str) and '|' in genres_str:
        return genres_str
    
    # 统一分隔符：将中文分隔符替换为标准分隔符
    # 先处理多种分隔符：／、中文逗号、英文逗号
    genres_str = genres_str.replace('／', '|')
    genres_str = genres_str.replace('，', '|')
    genres_str = genres_str.replace(',', '|')
    
    # 分割字符串，处理每个部分
    parts = [part.strip() for part in genres_str.split('|') if part.strip()]
    converted_parts = []
    
    for part in parts:
        # 检查是否为中文（在映射表中）
        if part in genres_mapping:
            converted_parts.append(genres_mapping[part])
        elif contains_chinese(part):
            # 如果包含中文但不在映射表中，保持原样
            converted_parts.append(part)
        else:
            # 不在映射表中，假设已经是英文，原样保留
            converted_parts.append(part)
    
    # 用|重新连接
    return '|'.join(converted_parts)

# 演员解析函数
def parse_actors(actor_str, delimiter=None):
    if pd.isna(actor_str) or actor_str == '':
        return None, None, None
    if delimiter:
        actors = str(actor_str).split(delimiter)
    else:
        actors = re.split(r'[,，/／|、\s]+', str(actor_str))
    actors = [a.strip() for a in actors if a.strip()]
    return (actors[0] if len(actors) > 0 else None,
            actors[1] if len(actors) > 1 else None,
            actors[2] if len(actors) > 2 else None)


In [ ]:
# movie2数据处理
def process_movie2(df):
    df_standard = pd.DataFrame(columns=standard_columns)
    if 'title' in df.columns:
        df_standard['movie_title'] = df['title']
    if 'type' in df.columns:
        df_standard['genres'] = df['type'].apply(convert_genres_to_english)
    if 'score' in df.columns:
        df_standard['imdb_score'] = df['score']
    if '票房' in df.columns:
        df_standard['gross'] = df['票房'].apply(convert_gross_to_usd)  # 关键：使用修正函数
    if 'showtime' in df.columns:
        df_standard['title_year'] = df['showtime']
    if 'area' in df.columns:
        df_standard['country'] = df['area']
    if '时长' in df.columns:
        df_standard['duration'] = df['时长']
    if 'actors' in df.columns:
        actors_parsed = df['actors'].apply(lambda x: parse_actors(x, '／'))
        df_standard['actor_1_name'] = [a[0] for a in actors_parsed]
        df_standard['actor_2_name'] = [a[1] for a in actors_parsed]
        df_standard['actor_3_name'] = [a[2] for a in actors_parsed]
    return df_standard

# movie3数据处理
def process_movie3(df):
    df_standard = pd.DataFrame(columns=standard_columns)
    if '电影名称' in df.columns:
        df_standard['movie_title'] = df['电影名称']
    if '上映日期' in df.columns:
        df_standard['title_year'] = df['上映日期']
    if '票房(万元)' in df.columns:
        # movie3是万元单位，需要先转元再转美元
        df_standard['gross'] = df['票房(万元)'].apply(
            lambda x: (x * 10000) / EXCHANGE_RATE if pd.notna(x) else np.nan
        )
    if '评分' in df.columns:
        df_standard['imdb_score'] = df['评分']
    if '评论数量' in df.columns:
        df_standard['num_user_for_reviews'] = df['评论数量']
    if '时长' in df.columns:
        df_standard['duration'] = df['时长']
    if '类型' in df.columns:
        df_standard['genres'] = df['类型'].apply(convert_genres_to_english)
    if '导演' in df.columns:
        df_standard['director_name'] = df['导演']
    if '演员表' in df.columns:
        actors_parsed = df['演员表'].apply(lambda x: parse_actors(x, ','))
        df_standard['actor_1_name'] = [a[0] for a in actors_parsed]
        df_standard['actor_2_name'] = [a[1] for a in actors_parsed]
        df_standard['actor_3_name'] = [a[2] for a in actors_parsed]
    return df_standard


In [ ]:
# 主处理流程
movie1_path = 'D:/bishe_zoujun/Flask/data_all/movie1.xlsx'
movie2_path = 'D:/bishe_zoujun/Flask/data_all/movie2.xlsx'
movie3_path = 'D:/bishe_zoujun/Flask/data_all/movie3.xlsx'

df1 = pd.read_excel(movie1_path)
df2 = pd.read_excel(movie2_path)
df3 = pd.read_excel(movie3_path)

print(f'movie1原始数据条数: {len(df1)}')
print(f'movie2原始数据条数: {len(df2)}')
print(f'movie3原始数据条数: {len(df3)}')

print('\n正在处理数据...')
df2_standard = process_movie2(df2)
df3_standard = process_movie3(df3)

merged_df = pd.concat([df1, df2_standard, df3_standard], ignore_index=True)


In [ ]:
# 国家名称转换：将中文国家名称转换为英文格式
if 'country' in merged_df.columns:
    print('\n正在转换国家名称（中文→英文）...')
    merged_df['country'] = merged_df['country'].apply(convert_country_to_english)
    print('国家名称转换完成！')

# 电影类型转换：统一为英文格式（|分隔）
if 'genres' in merged_df.columns:
    print('\n正在统一电影类型格式（中文→英文）...')
    merged_df['genres'] = merged_df['genres'].apply(convert_genres_to_english)
    print('电影类型转换完成！')

# 数据去重：以movie_title为标准，保留第一条记录
if 'movie_title' in merged_df.columns:
    print('\n正在去重（movie_title为标准）...')
    before_deduplicate = len(merged_df)
    merged_df = merged_df.drop_duplicates(subset=['movie_title'], keep='first')
    after_deduplicate = len(merged_df)
    removed_count = before_deduplicate - after_deduplicate
    print(f'去重完成！移除 {removed_count} 条重复记录')
    print(f'去重前: {before_deduplicate} 条')
    print(f'去重后: {after_deduplicate} 条')

print('=' * 50)
print('数据合并完成！')
print('=' * 50)
print(f'合并后总数据条数: {len(merged_df)}')
print('=' * 50)

# 保存结果
output_excel = 'D:/bishe_zoujun/test1/data_all/movies_all_cleaned.xlsx'
output_csv = 'D:/bishe_zoujun/test1/data_all/movies_all_cleaned.csv'
merged_df.to_excel(output_excel, index=False)
merged_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'\n文件已保存:')
print(f'  Excel文件: {output_excel}')
print(f'  CSV文件: {output_csv}')

movie1原始数据条数: 4458
movie2原始数据条数: 1069
movie3原始数据条数: 3227

正在处理数据...

正在转换国家名称（中文→英文）...
国家名称转换完成！

正在统一电影类型格式（中文→英文）...


C:\Users\24879\AppData\Local\Temp\ipykernel_18392\3965737835.py:369: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_df = pd.concat([df1, df2_standard, df3_standard], ignore_index=True)


电影类型转换完成！

正在去重（movie_title为标准）...
去重完成！移除 1056 条重复记录
去重前: 8754 条
去重后: 7698 条
数据合并完成！
合并后总数据条数: 7698

文件已保存:
  Excel文件: D:/bishe_zoujun/test1/movies_all_cleaned.xlsx
  CSV文件: D:/bishe_zoujun/test1/movies_all_cleaned.csv
